# Reproject & Mask Woody Cover — Preprocessing Step 2

**Version:** 1.0 (2026-03-09) © R. Butzer

## What this notebook does
1. Reprojects wildE Woody Cover VRT (30m, native CRS) → 500m, EPSG:3035
   - Grid parameters (extent, resolution, CRS) are read from the MBA binary mask (output of `01_export_fire_metrics.ipynb`)
2. Masks the reprojected raster to the European landmass extent

## Inputs
- `A:\_BioGeo\wildE\_PREDICTIONS\Run03_April2025\woody_cover.vrt` — wildE Woody Cover (all years)
- `_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_binary_yearly_500m.tif` — Grid template (from `01_export_fire_metrics.ipynb`)
- `_00_SHAPEFILES\europe_landmass.gpkg` — Europe landmass polygon

## Outputs
- `_Runs\02_Woody_Cover\woody_cover_500m_3035.tif` — Reprojected woody cover (500m, EPSG:3035)
- `_Runs\02_Woody_Cover\woody_cover_500m_3035_europe_masked.tif` — Europe-masked version → **Input for `03_combine_woody_fire.ipynb`**

In [1]:
from osgeo import gdal
import time
import os
import rasterio
from rasterio import mask
import fiona
import numpy as np

In [4]:
woody_vrt = r"A:\_BioGeo\wildE\_PREDICTIONS\Run03_April2025\woody_cover.vrt"
workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
burned_mask_path = os.path.join(
    workDir,
    r"_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_binary_yearly_500m.tif"
)
out_woody_path = os.path.join(
    workDir,
    r"_Runs\02_Woody_Cover\woody_cover_500m_3035.tif"
)
europe_gpkg = os.path.join(
    workDir,
    r"_00_SHAPEFILES\europe_landmass.gpkg"
)
out_masked = out_woody_path.replace('.tif', '_europe_masked.tif')
nodata_value = 255  # uint8

os.makedirs(os.path.join(workDir, r"_Runs\02_Woody_Cover"), exist_ok=True)


In [5]:

# --- Step 1: Read grid parameters from MBA binary mask ---
print("Lese Zielraster-Parameter von der 500m Burned Mask ...")
mask_ds = gdal.Open(burned_mask_path)
gt = mask_ds.GetGeoTransform()
proj = mask_ds.GetProjection()
x_size = mask_ds.RasterXSize
y_size = mask_ds.RasterYSize
mask_ds = None  # close dataset

# --- Step 2: Reproject wildE VRT → 500m EPSG:3035 ---
print("Starte Reprojektion mit GDAL ...")
start = time.time()
gdal.Warp(
    out_woody_path,
    woody_vrt,
    format='GTiff',
    outputBounds=(gt[0], gt[3] + y_size * gt[5], gt[0] + x_size * gt[1], gt[3]),
    xRes=gt[1],
    yRes=abs(gt[5]),
    dstSRS=proj,
    resampleAlg='near',
    multithread=True,
    options=['COMPRESS=LZW']
)
print(f"✓ Reprojektion abgeschlossen. Dauer: {time.time() - start:.1f} Sekunden")
print(f"  Output: {out_woody_path}")

# --- Step 3: Mask to European landmass ---
print("\nLade Europe Landmass Polygon ...")
with fiona.open(europe_gpkg, layer=0) as shapefile:
    geoms = [feature["geometry"] for feature in shapefile]

print("Maskiere außerhalb Europas ...")
start = time.time()
with rasterio.open(out_woody_path) as src:
    out_image, out_transform = mask.mask(src, geoms, crop=False, nodata=nodata_value)
    out_meta = src.meta.copy()
print(f"✓ Maskierung abgeschlossen. Dauer: {time.time() - start:.2f} Sekunden")

# Set pixels with all-zero values across bands to nodata
zero_mask = np.all(out_image == 0, axis=0)
out_image[:, zero_mask] = nodata_value

out_meta.update({
    "driver": "GTiff",
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform,
    "nodata": nodata_value
})

print("Speichere maskiertes Raster ...")
start = time.time()
with rasterio.open(out_masked, "w", **out_meta) as dest:
    dest.write(out_image)
print(f"✓ Gespeichert. Dauer: {time.time() - start:.2f} Sekunden")

print("\n" + "="*60)
print("✓✓✓ PREPROCESSING STEP 1 ABGESCHLOSSEN ✓✓✓")
print("="*60)
print(f"  Output (reprojected): {out_woody_path}")
print(f"  Output (masked):      {out_masked}")
print("  → Weiter mit: 03_combine_woody_fire.ipynb")
print("="*60)

Lese Zielraster-Parameter von der 500m Burned Mask ...
Starte Reprojektion mit GDAL ...
✓ Reprojektion abgeschlossen. Dauer: 8805.1 Sekunden
  Output: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Woody_Cover\woody_cover_500m_3035.tif

Lade Europe Landmass Polygon ...
Maskiere außerhalb Europas ...
✓ Maskierung abgeschlossen. Dauer: 114.04 Sekunden
Speichere maskiertes Raster ...
✓ Gespeichert. Dauer: 82.50 Sekunden

✓✓✓ PREPROCESSING STEP 1 ABGESCHLOSSEN ✓✓✓
  Output (reprojected): \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Woody_Cover\woody_cover_500m_3035.tif
  Output (masked):      \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\02_Woody_Cover\woody_cover_500m_3035_europe_masked.tif
  → Weiter mit: 03_combine_woody_fire.ipynb
